# 🏢 HVAC-RL 完整Pipeline (A100优化版)

## 📋 说明
- ✅ **GPU要求**: A100 (推荐) 或 V100
- ⏱️ **预计时间**: 4-8小时（完整pipeline）
- 💾 **存储需求**: ~15GB

## 🚀 Pipeline流程
```
1. PPO训练 (收集高质量轨迹)
    ↓
2. Few-shot示例选择 (筛选多样化样本)
    ↓
3. LLM Rollout (生成自己的轨迹)
    ↓
4. 自我蒸馏数据准备 (筛选高reward数据)
    ↓
5. LoRA Fine-tuning (使用自我蒸馏数据)
    ↓
6. 评估和对比
```

---
## ⚙️ Step 1: 环境设置

In [ ]:
# 1.1 检查GPU
import torch
import os

print("=" * 60)
print("GPU信息")
print("=" * 60)
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU显存: {gpu_mem:.1f} GB")
    
    if gpu_mem < 30:
        print("⚠️  警告: 显存小于30GB，建议使用A100 (40GB)")
    else:
        print("✅ GPU配置充足！")
else:
    print("❌ 未检测到GPU！请检查Runtime设置")
    print("   Runtime > Change runtime type > Hardware accelerator > GPU")

In [ ]:
# 1.2 挂载Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 创建工作目录
WORK_DIR = '/content/drive/MyDrive/HVAC_RL_Results'
os.makedirs(WORK_DIR, exist_ok=True)
print(f"✅ Google Drive已挂载")
print(f"✅ 工作目录: {WORK_DIR}")

In [ ]:
# 1.3 克隆项目
PROJECT_DIR = "/content/HVAC-RL"

if os.path.exists(PROJECT_DIR):
    print("项目已存在，拉取最新更新...")
    !cd {PROJECT_DIR} && git pull
else:
    print("克隆项目...")
    !git clone https://github.com/Mo119m/HAVC-control-with-reinforcement-learning-update.git {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f"✅ 当前目录: {os.getcwd()}")
print(f"✅ 项目文件:")
!ls -la

In [ ]:
# 1.4 安装依赖
print("安装依赖包...")
!pip install -q torch transformers accelerate peft
!pip install -q stable-baselines3 sb3-contrib gymnasium
!pip install -q numpy pandas scikit-learn scipy matplotlib
!pip install -q pvlib cvxpy tqdm

print("\n✅ 所有依赖已安装！")

# 验证安装
import gymnasium
import stable_baselines3
import transformers
print(f"\n版本检查:")
print(f"  - gymnasium: {gymnasium.__version__}")
print(f"  - stable-baselines3: {stable_baselines3.__version__}")
print(f"  - transformers: {transformers.__version__}")

In [ ]:
# 1.5 验证环境
print("运行环境验证...\n")
!python verify_environment.py

---
## 🎯 Step 2: 配置Pipeline参数

In [ ]:
# 配置参数
CONFIG = {
    # 建筑和气候
    "building": "OfficeSmall",  # 可选: OfficeSmall, OfficeMedium, OfficeLarge等
    "weather": "Hot_Dry",
    "location": "Tucson",
    
    # PPO训练
    "ppo_steps": 500000,  # 完整训练: 500000, 快速测试: 10000
    
    # LLM配置
    "model_name": "Qwen/Qwen2.5-7B-Instruct",
    "temperature": 0.7,
    
    # Fine-tuning
    "finetune_epochs": 3,
    "finetune_lr": 1e-4,
    "batch_size": 4,  # A100可以用4，V100用2
    
    # 输出目录
    "output_dir": "/content/pipeline_output",
}

# 创建输出目录
os.makedirs(CONFIG["output_dir"], exist_ok=True)

print("=" * 60)
print("Pipeline配置")
print("=" * 60)
for key, value in CONFIG.items():
    print(f"{key:20s}: {value}")
print("=" * 60)

---
## 🚀 Step 3: 运行完整Pipeline

### 选项A: 一键运行所有阶段 (推荐)

In [ ]:
# 运行完整Pipeline (所有6个阶段)
import time

start_time = time.time()

print("🚀 开始运行完整Pipeline...\n")
print("这将依次执行:")
print("  1. PPO训练")
print("  2. Few-shot示例选择")
print("  3. LLM Rollout")
print("  4. 自我蒸馏数据准备")
print("  5. LoRA Fine-tuning")
print("  6. 评估和对比")
print("\n预计耗时: 4-8小时 (A100)\n")

!python core_modules/main_pipeline.py \
    --stage all \
    --building {CONFIG["building"]} \
    --weather {CONFIG["weather"]}

elapsed = (time.time() - start_time) / 3600
print(f"\n✅ Pipeline完成！总耗时: {elapsed:.2f} 小时")

### 选项B: 分阶段运行 (调试时使用)

In [ ]:
# Stage 1: PPO训练
print("📊 Stage 1: PPO训练\n")
!python core_modules/main_pipeline.py \
    --stage ppo \
    --building {CONFIG["building"]} \
    --weather {CONFIG["weather"]}

In [ ]:
# Stage 2: Few-shot示例选择
print("🎯 Stage 2: Few-shot示例选择\n")
!python core_modules/main_pipeline.py --stage select

In [ ]:
# Stage 3: LLM Rollout
print("🤖 Stage 3: LLM Rollout\n")
!python core_modules/main_pipeline.py --stage rollout

In [ ]:
# Stage 4: 自我蒸馏数据准备
print("🔬 Stage 4: 自我蒸馏数据准备\n")
!python core_modules/main_pipeline.py --stage distill

In [ ]:
# Stage 5: LoRA Fine-tuning
print("🎓 Stage 5: LoRA Fine-tuning\n")
!python core_modules/main_pipeline.py --stage finetune

In [ ]:
# Stage 6: 评估
print("📈 Stage 6: 评估和对比\n")
!python core_modules/main_pipeline.py --stage eval

---
## 📊 Step 4: 查看结果

In [ ]:
# 检查输出文件
import os
from pathlib import Path

output_dir = Path("./pipeline_output")

print("=" * 60)
print("Pipeline输出文件")
print("=" * 60)

stages = [
    ("01_ppo_training", ["ppo_trajectory.json", "ppo_final.zip", "ppo_training_results.png"]),
    ("02_few_shot_samples", ["few_shot_examples_structured.json"]),
    ("03_llm_rollout", ["llm_rollout.json", "distillation_data.json"]),
    ("04_finetuning", ["final_model/"]),
    ("05_evaluation", ["finetuned_rollout.json", "comparison_plot.png"]),
]

for stage_dir, files in stages:
    stage_path = output_dir / stage_dir
    print(f"\n{stage_dir}:")
    if stage_path.exists():
        for f in files:
            file_path = stage_path / f
            if file_path.exists():
                if file_path.is_dir():
                    print(f"  ✅ {f} (目录)")
                else:
                    size = file_path.stat().st_size / 1024 / 1024
                    print(f"  ✅ {f} ({size:.2f} MB)")
            else:
                print(f"  ⏳ {f} (未生成)")
    else:
        print(f"  ❌ 目录不存在")

In [ ]:
# 显示对比图
import matplotlib.pyplot as plt
from PIL import Image

comparison_plot = "./pipeline_output/05_evaluation/comparison_plot.png"

if os.path.exists(comparison_plot):
    img = Image.open(comparison_plot)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('PPO vs LLM vs Fine-tuned LLM', fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("❌ 对比图未找到，请先运行Stage 6")

In [ ]:
# 计算统计数据
import json
import numpy as np

def load_trajectory(path):
    if not os.path.exists(path):
        return None
    with open(path, 'r') as f:
        return json.load(f)

def get_rewards(traj):
    if isinstance(traj, list):
        return [step.get('reward', 0) for step in traj]
    return []

# 加载轨迹
ppo_traj = load_trajectory('./pipeline_output/01_ppo_training/ppo_trajectory.json')
llm_traj = load_trajectory('./pipeline_output/03_llm_rollout/llm_rollout.json')
ft_traj = load_trajectory('./pipeline_output/05_evaluation/finetuned_rollout.json')

print("=" * 60)
print("性能对比")
print("=" * 60)

if ppo_traj:
    ppo_rewards = get_rewards(ppo_traj)
    if ppo_rewards:
        print(f"\nPPO Expert (baseline):")
        print(f"  平均reward: {np.mean(ppo_rewards):.2f}")
        print(f"  总reward: {np.sum(ppo_rewards):.2f}")
        print(f"  步数: {len(ppo_rewards)}")

if llm_traj:
    llm_rewards = get_rewards(llm_traj)
    if llm_rewards:
        print(f"\nLLM Before Fine-tuning:")
        print(f"  平均reward: {np.mean(llm_rewards):.2f}")
        print(f"  总reward: {np.sum(llm_rewards):.2f}")
        print(f"  步数: {len(llm_rewards)}")

if ft_traj:
    ft_rewards = get_rewards(ft_traj)
    if ft_rewards:
        print(f"\nLLM After Fine-tuning:")
        print(f"  平均reward: {np.mean(ft_rewards):.2f}")
        print(f"  总reward: {np.sum(ft_rewards):.2f}")
        print(f"  步数: {len(ft_rewards)}")
        
        if llm_rewards:
            improvement = (np.mean(ft_rewards) - np.mean(llm_rewards)) / abs(np.mean(llm_rewards)) * 100
            print(f"\n📈 Fine-tuning提升: {improvement:+.1f}%")

print("=" * 60)

---
## 💾 Step 5: 保存到Google Drive

In [ ]:
# 复制所有结果到Google Drive
import shutil
from datetime import datetime

# 创建带时间戳的目录
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = f"/content/drive/MyDrive/HVAC_RL_Results/run_{timestamp}"
os.makedirs(save_dir, exist_ok=True)

print(f"保存结果到: {save_dir}\n")

# 复制整个pipeline_output目录
if os.path.exists("./pipeline_output"):
    shutil.copytree("./pipeline_output", f"{save_dir}/pipeline_output", dirs_exist_ok=True)
    print("✅ Pipeline输出已保存")

# 保存配置
with open(f"{save_dir}/config.json", 'w') as f:
    json.dump(CONFIG, f, indent=2)
print("✅ 配置已保存")

# 显示目录结构
print(f"\n保存的文件:")
!ls -lh {save_dir}

print(f"\n✅ 所有结果已保存到Google Drive!")
print(f"📂 位置: {save_dir}")

---
## 🔧 Step 6: 故障排除和工具

In [ ]:
# 监控GPU使用情况
!nvidia-smi

In [ ]:
# 检查磁盘空间
!df -h | grep -E 'Filesystem|/content'

In [ ]:
# 清理缓存（如果空间不足）
import torch
import gc

# 清理GPU缓存
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ GPU缓存已清理")

# 清理Python垃圾回收
gc.collect()
print("✅ 内存已清理")

In [ ]:
# 快速恢复（Colab断开后使用）
from google.colab import drive
import os

# 重新挂载Drive
drive.mount('/content/drive', force_remount=True)

# 切换到项目目录
PROJECT_DIR = "/content/HVAC-RL"
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    print(f"✅ 已切换到项目目录: {os.getcwd()}")
else:
    print("❌ 项目不存在，请重新运行环境设置")

# 检查已完成的stage
if os.path.exists("./pipeline_output"):
    print("\n已完成的stages:")
    for stage in os.listdir("./pipeline_output"):
        print(f"  ✅ {stage}")
else:
    print("\n❌ 未找到pipeline输出，需要从头开始")

---
## 📝 说明

### 预计运行时间（A100 40GB）
- Stage 1 (PPO训练): 1-2小时
- Stage 2 (Few-shot选择): 5-10分钟
- Stage 3 (LLM Rollout): 30-60分钟
- Stage 4 (自我蒸馏): 5分钟
- Stage 5 (Fine-tuning): 2-4小时
- Stage 6 (评估): 30分钟

**总计**: 约4-8小时

### 如何加快速度
1. 减少PPO训练步数（`ppo_steps`）
2. 减少Fine-tuning epochs
3. 使用更小的模型（Qwen2.5-1.8B）

### 常见问题

**Q: CUDA out of memory**
- 减小batch_size到2或1
- 重启runtime清理内存

**Q: Colab断开连接**
- 运行"快速恢复"cell
- 从断开的stage继续运行

**Q: 如何使用不同建筑/气候**
- 修改CONFIG中的`building`和`weather`
- 可用建筑: OfficeSmall, OfficeMedium, OfficeLarge, Hospital等
- 可用气候: Hot_Dry, Cold_Humid, Warm_Marine等